In [3]:
#import and setup

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import random_split,TensorDataset,DataLoader,Subset
from sklearn.metrics import precision_score, recall_score, f1_score,confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
#Load and inspect the dataset

df = pd.read_csv(r"D:\Codes\Deep learning\Predictive Maintenance AI\data\ai4i2020.csv")


print("Shape : \n",df.shape)
print("\nData types : ",df.dtypes)
print("\nMissing values ",df.isnull().sum())
print("\nDuplicate rows ",df.duplicated().sum())
print("\nDistribution of failure :",df["Machine failure"].value_counts())
print("\n\nDistribution of type :",df["Type"].value_counts())
print("\nStatistical summary:")
print(df.describe())

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Codes\\Deep learning\\Predictive Maintenance AI\\data\\ai4i2020.csv'

In [ ]:
#exploratory data analysis (EDA)

### Failure vs. Non-Failure Analysis

df.groupby("Machine failure")["Air temperature [K]"].mean() # Machines that failed had a slightly  0.91 K higher average air temperature
df.groupby("Machine failure")["Process temperature [K]"].mean() #Machines that failed had a slightly 0.29 K higher average process temperature
df.groupby("Machine failure")["Rotational speed [rpm]"].mean() #machines that failed had lower 43.77 rpm average rotational speed
df.groupby("Machine failure")["Torque [Nm]"].mean() #big difference of about 10.54 Nm
df.groupby("Machine failure")["Tool wear [min]"].mean() #difference of about 37.09 minutes
df.groupby("Machine failure")["TWF"].mean() #13.57% of failed machines had TWF = 1
df.groupby("Machine failure")["HDF"].mean() #33.92% of failed machines had HDF = 1
df.groupby("Machine failure")["PWF"].mean() #28.02% of failed machines had PWF = 1
df.groupby("Machine failure")["OSF"].mean() #28.91% of failed machines had OSF = 1
df.groupby("Machine failure")["RNF"].mean() #RNF has only a very small difference


In [ ]:
#converting L, M, H to one-hot encoding
df = pd.get_dummies(df, columns=["Type"], dtype=int)
print(df.columns)

In [ ]:
#Feature Engineering 

x = df[["Air temperature [K]","Process temperature [K]","Rotational speed [rpm]","Torque [Nm]","Tool wear [min]",'Type_H',
       'Type_L', 'Type_M']]
y = df["Machine failure"]

feature_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_H",
    "Type_L",
    "Type_M"
]
print(x.shape)
print(y.shape)


x = torch.tensor(x.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)

In [ ]:
# Train / Validation / Test Split

torch.manual_seed(42)
dataset = TensorDataset(x,y)

train_size = 7000
val_size = 1500
test_size = 1500


train_data,val_data,test_data = random_split(
    dataset,
    [train_size,val_size,test_size]
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size = 32,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size = 32,
    shuffle=False
)

In [ ]:
#feature scaling

x_train = x[train_data.indices]

mean = x_train[:,:5].mean(dim=0)
std = x_train[:,:5].std(dim=0)

#scaling the entire data
x_scaled = x.clone()
x_scaled[:, :5] = (x[:, :5] - mean) / std

print("Scaled mean:", mean)
print("Scaled std:", std)


In [ ]:
dataset = TensorDataset(x_scaled,y)
train_data = Subset(dataset,train_data.indices)
val_data = Subset(dataset,val_data.indices)
test_data = Subset(dataset,test_data.indices)
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
print("Train:", sum(y[i].item() for i in train_data.indices))
print("Validation:", sum(y[i].item() for i in val_data.indices))
print("Test:", sum(y[i].item() for i in test_data.indices))

In [ ]:
# handle class imbalance

train_targets = y[train_data.indices]

N = len(train_targets)

N0 = (train_targets == 0).sum().item()
N1 = (train_targets == 1).sum().item()

weight_0 = N / (2 * N0)
weight_1 = N / (2 * N1)

class_weight = torch.tensor(
    [weight_0, weight_1],
    dtype=torch.float32
)


In [ ]:
#Build the neural network

class_weight = class_weight.to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weight)
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(8,16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,2)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.01,
    weight_decay=0.001
)

patience = 10
count = 0
best_val_loss = float("inf")

In [ ]:
#train the model

for epoch in range(100):

    total_train_loss = 0

    model.train()

    for batch_x,batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        prediction = model(batch_x)
        loss = loss_fn(prediction,batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    total_val_loss= 0

    model.eval()

    with torch.no_grad():
        for batch_x,batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            prediction = model(batch_x)
            loss = loss_fn(prediction,batch_y)
            total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss/len(val_loader)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
            "model_state_dict": model.state_dict(),
            "mean": mean,
            "std": std,
            "feature_columns": feature_columns
            }, "../model/best_model.pth")
            count = 0

        else:
            count+=1


    # Print every 20 epochs
    if (epoch + 1) % 20 == 0:

        print(
            f"Epoch {epoch + 1} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"val Loss: {avg_val_loss:.4f}"
        )

    if count >= patience:
        print(f"Epoch : {epoch +1} Early stop, at best val loss {best_val_loss} ")
        break

# Load the best model saved during training
checkpoint = torch.load(
    "../model/best_model.pth",
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(device)
model.eval()

In [ ]:
#threshold selection

model.eval()

all_probs = []
all_targets = []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)

        probabilities = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probabilities.cpu().numpy())
        all_targets.extend(batch_y.numpy())


# Try different thresholds
thresholds = [i / 100 for i in range(50, 81)]

for threshold in thresholds:

    predictions = [
        1 if prob >= threshold else 0
        for prob in all_probs
    ]

    precision = precision_score(all_targets, predictions, zero_division=0)
    recall = recall_score(all_targets, predictions, zero_division=0)
    f1 = f1_score(all_targets, predictions, zero_division=0)

    # Calculate FP and FN
    fp = sum(
        1 for actual, predicted in zip(all_targets, predictions)
        if actual == 0 and predicted == 1
    )

    fn = sum(
        1 for actual, predicted in zip(all_targets, predictions)
        if actual == 1 and predicted == 0
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f} | "
        f"FP: {fp} | "
        f"FN: {fn}"
    )

### Threshold Selection

The default classification threshold of 0.5 is not necessarily optimal for an imbalanced predictive-maintenance problem. Since missing an actual machine failure can be more costly than generating a false alarm, different thresholds are evaluated on the validation set.

Precision, recall, F1-score, false positives (FP), and false negatives (FN) are compared to identify an appropriate operating threshold.


In [ ]:
#update checkpoint 

checkpoint = torch.load(
    "../model/best_model.pth",
    weights_only=True
)

checkpoint["threshold"] = 0.61

torch.save(
    checkpoint,
    "../model/best_model.pth"
)

In [ ]:
#final test evaluation

model.eval()

all_probs = []
all_targets = []

# Get class-1 probabilities from TEST set
with torch.no_grad():
    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)

        logits = model(batch_x)

        probabilities = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probabilities.cpu().numpy())
        all_targets.extend(batch_y.numpy())


# Apply the threshold selected using the validation set
threshold = 0.61

predictions = [
    1 if prob >= threshold else 0
    for prob in all_probs
]

# Metrics
cm = confusion_matrix(all_targets, predictions)

precision = precision_score(
    all_targets, predictions, zero_division=0
)

recall = recall_score(
    all_targets, predictions, zero_division=0
)

f1 = f1_score(
    all_targets, predictions, zero_division=0
)

# Extract confusion matrix values
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print(f"\nTrue Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

### Final Test Evaluation

The test set is used only for the final evaluation of the trained model. The classification threshold was selected using the validation set and fixed at **0.61** before evaluating the test data.

The final model is evaluated using the confusion matrix, precision, recall, and F1-score. Because machine failures are the minority class, recall is particularly important for measuring how effectively the model detects actual failures.


In [ ]:
# Single Machine Inference

checkpoint = torch.load(
    "../model/best_model.pth",
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

mean = checkpoint["mean"]
std = checkpoint["std"]
threshold = checkpoint["threshold"]
feature_columns = checkpoint["feature_columns"]



new_data = {"Air temperature [K]" : 298.2,
"Process temperature [K]" : 309.4,
"Rotational speed [rpm]"  : 1439,
"Torque [Nm]"             : 37.3,
"Tool wear [min]"         : 58,
"Type"                     : "H"}

new_data = pd.DataFrame([new_data])
new_data = pd.get_dummies(new_data, columns=["Type"], dtype=int)
new_data = new_data.reindex(columns=feature_columns, fill_value=0)
one_hot = new_data[["Type_H", "Type_L", "Type_M"]]
scale = new_data.iloc[:, :5]
scaled_data = (scale-mean)/std

x = pd.concat([scaled_data, one_hot], axis=1)
print(x)
x_tensor = torch.tensor(x.values,dtype =torch.float32)

x_tensor = x_tensor.to(device)

model.eval()
with torch.no_grad():
    prediction = model(x_tensor)
    probabilities = torch.softmax(prediction, dim=1)
    failure_probability = probabilities[0,1].item()

threshold = 0.61

if failure_probability >= threshold:
    print("Machine Failure: YES")
    print(f"Failure Probability {probabilities[0,1].item()*100:.1f}%")
else:
    print("Machine Failure: NO")
    print(f"Failure Probability {probabilities[0,0].item()*100:.1f}%")
